# End-to-End Pipeline: Random Forest → KMeans → XGBoost (Full Tuning)

This notebook runs the full project in three stages:
1) Random Forest (rank-adapted)
2) KMeans clustering (model selection, stability, profiles)
3) XGBoost learning-to-rank with full Optuna tuning

Artifacts are saved under `outputs/` and models under `models/`.



In [ ]:
# Setup & Environment
import os, sys, json, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# ML packages
import sklearn
import matplotlib.pyplot as plt

# Ensure src on path
sys.path.append(os.path.abspath('.'))

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Dirs
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs/plots', exist_ok=True)

print('Python:', sys.version)
print('sklearn:', sklearn.__version__)



In [ ]:
# 2) Data Preparation
# 2a) Rebuild core features (writes outputs/data_prepared.pkl)
import runpy
_ = runpy.run_path('data_preparation.py')

# 2b) Build ranking dataset (writes outputs/data_prepared_rank.pkl)
from src.ranking.prepare import main as build_ranking_dataset
build_ranking_dataset()

import pickle
with open('outputs/data_prepared_rank.pkl', 'rb') as f:
    rank_meta = pickle.load(f)
print('Ranking data shapes:',
      rank_meta['X_train'].shape,
      rank_meta['X_val'].shape,
      rank_meta['X_test'].shape)



In [ ]:
# 3) Random Forest (rank-adapted)
from sklearn.ensemble import RandomForestRegressor
from src.ranking.metrics import groupwise_eval
import json

with open('outputs/data_prepared_rank.pkl', 'rb') as f:
    data = pickle.load(f)

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
G_val, G_test = data['group_val'], data['group_test']

rf = RandomForestRegressor(
    n_estimators=600,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=SEED,
)
rf.fit(X_train, y_train)

val_preds = rf.predict(X_val)
test_preds = rf.predict(X_test)

val_metrics = groupwise_eval(y_val, val_preds, G_val)
test_metrics = groupwise_eval(y_test, test_preds, G_test)

art = {
    'validation': val_metrics,
    'test': test_metrics,
}
with open('outputs/rf_rank_metrics.json', 'w') as f:
    json.dump(art, f, indent=2)

# Export top-10 per student (test)
rows, idx, cursor = [], 0, 0
tsid = data.get('test_student_id', [])
tsport = data.get('test_sport', [])
for g in G_test:
    scores = test_preds[idx: idx + g]
    sid_slice = tsid[cursor: cursor + g] if tsid else [''] * g
    sport_slice = tsport[cursor: cursor + g] if tsport else [''] * g
    order = np.argsort(-scores)[:min(10, g)]
    for rpos, li in enumerate(order, start=1):
        rows.append({
            'student_id': sid_slice[li],
            'sport': sport_slice[li],
            'rank_position': int(rpos),
            'predicted_score': float(scores[li]),
        })
    idx += g
    cursor += g
pd.DataFrame(rows).to_csv('outputs/rf_rank_predictions_top10.csv', index=False)

import joblib
joblib.dump(rf, 'models/rf_rank.pkl')
print('RF rank-adapted completed. Test NDCG@10:', test_metrics.get('ndcg@10'))



In [ ]:
# 4) KMeans Clustering (improved)
from src.clustering.model import main as clustering_main
clustering_main()

import json
with open('outputs/clustering_optimal_model.json', 'r') as f:
    best = json.load(f)
print('Best clustering:', best['algorithm'], best['params'])



In [ ]:
# 5) XGBoost Ranker (Full Optuna Tuning)
import optuna
from xgboost import XGBRanker
from src.ranking.metrics import groupwise_eval

TRIALS = 100  # full tuning

with open('outputs/data_prepared_rank.pkl', 'rb') as f:
    d = pickle.load(f)

Xt, yt, Gt = d['X_train'], d['y_train'], d['group_train']
Xv, yv, Gv = d['X_val'], d['y_val'], d['group_val']
Xs, ys, Gs = d['X_test'], d['y_test'], d['group_test']

def objective(trial):
    params = {
        'objective': 'rank:ndcg',
        'eval_metric': 'ndcg@10',
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 12),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 3.0),
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': 0,
    }
    model = XGBRanker(**params)
    model.fit(Xt, yt, group=Gt, eval_set=[(Xv, yv)], eval_group=[Gv], verbose=False)
    preds = model.predict(Xv)
    m = groupwise_eval(yv, preds, Gv)
    return m['ndcg@10']

study = optuna.create_study(direction='maximize', study_name='xgb_ranker_full')
study.optimize(objective, n_trials=TRIALS, show_progress_bar=True)

best_params = {'objective': 'rank:ndcg', 'eval_metric': 'ndcg@10', 'random_state': SEED, 'n_jobs': -1, 'verbosity': 0}
best_params.update(study.best_trial.params)

best_model = XGBRanker(**best_params)
best_model.fit(Xt, yt, group=Gt, eval_set=[(Xv, yv)], eval_group=[Gv], verbose=False)

val_preds = best_model.predict(Xv)
test_preds = best_model.predict(Xs)
val_m = groupwise_eval(yv, val_preds, Gv)
test_m = groupwise_eval(ys, test_preds, Gs)

with open('outputs/ranking_optimization_results.json', 'w') as f:
    json.dump({'best_params': study.best_trial.params, 'best_ndcg10': float(study.best_value)}, f, indent=2)
with open('outputs/ranking_metrics_tuned.json', 'w') as f:
    json.dump({'validation': val_m, 'test': test_m, 'best_params': best_params}, f, indent=2)

# Export top-10 predictions
rows, idx, cursor = [], 0, 0
tsid = d.get('test_student_id', [])
tsport = d.get('test_sport', [])
for g in Gs:
    scores = test_preds[idx: idx + g]
    sid_slice = tsid[cursor: cursor + g] if tsid else [''] * g
    sport_slice = tsport[cursor: cursor + g] if tsport else [''] * g
    order = np.argsort(-scores)[:min(10, g)]
    for rpos, li in enumerate(order, start=1):
        rows.append({
            'student_id': sid_slice[li],
            'sport': sport_slice[li],
            'rank_position': int(rpos),
            'predicted_score': float(scores[li]),
        })
    idx += g
    cursor += g
pd.DataFrame(rows).to_csv('outputs/ranking_predictions_top10_tuned.csv', index=False)

import joblib
joblib.dump(best_model, 'models/xgb_ranker_tuned.pkl')
print('Tuned XGB complete. Test NDCG@10:', test_m.get('ndcg@10'))



In [ ]:
# 6) Promote Best Model to Production
import shutil
src = 'models/xgb_ranker_tuned.pkl'
dst = 'models/xgb_ranker_production.pkl'
shutil.copyfile(src, dst)

from datetime import datetime
meta = {
    'source': src,
    'destination': dst,
    'promoted_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'notes': 'Promoted from notebook full tuning run.'
}
with open('outputs/ranker_production_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Promoted tuned model → production.')



In [ ]:
# 7) Batch Inference Demo (Production Model)
# Build a small synthetic features CSV from the validation base (without sport one-hots)
X_cols = d['X_columns']
base_cols = [c for c in X_cols if not c.startswith('sport_') and c not in ('student_id','rank_idx','relevance','score')]

# We'll take first 5 rows from validation set, reconstruct base features by reading from
# the original features DataFrame used to create ranking inputs is not directly stored.
# As a demo, we create a DataFrame with zeros and fill with column means from training.
X_base = pd.DataFrame(0.0, index=range(5), columns=base_cols)

# Estimate means from training slice by projecting back columns present
# (Here we approximate by using column-wise medians derived from X_train matrix whenever names match)
# Note: Since X_train includes sport one-hots, we only set base columns to 0 by default in this demo.
X_base.to_csv('outputs/demo_inference_features.csv', index=False)

# Run inference assembling candidate sports
import pickle
import numpy as np
with open('outputs/data_prepared_rank.pkl', 'rb') as f:
    meta = pickle.load(f)

catalog = meta['catalog_sports']
X_columns = meta['X_columns']
sport_ohe_cols = [c for c in X_columns if c.startswith('sport_')]
base_cols = [c for c in X_columns if c not in sport_ohe_cols and c not in ('student_id','rank_idx','relevance','score')]

from xgboost import XGBRanker
import joblib
prod = joblib.load('models/xgb_ranker_production.pkl')

rows = []
for idx in range(len(X_base)):
    x_row = X_base.iloc[idx:idx+1].to_numpy()
    X_rep = np.repeat(x_row, len(catalog), axis=0)
    sport_matrix = np.zeros((len(catalog), len(sport_ohe_cols)), dtype=float)
    col_index = {c: j for j, c in enumerate(sport_ohe_cols)}
    for i, sport in enumerate(catalog):
        col_name = f'sport_{sport}'
        j = col_index.get(col_name, None)
        if j is not None:
            sport_matrix[i, j] = 1.0
    X_full = np.zeros((len(catalog), len(X_columns)), dtype=float)
    base_index = {c: j for j, c in enumerate(base_cols)}
    for j, col in enumerate(X_columns):
        if col in base_index:
            X_full[:, j] = X_rep[:, base_index[col]]
        elif col in col_index:
            X_full[:, j] = sport_matrix[:, col_index[col]]
        else:
            X_full[:, j] = 0.0
    scores = prod.predict(X_full)
    order = np.argsort(-scores)[:10]
    for rpos, i in enumerate(order, start=1):
        rows.append({'row_index': idx, 'sport': catalog[i], 'rank_position': rpos, 'predicted_score': float(scores[i])})

pd.DataFrame(rows).to_csv('outputs/ranking_inference_top10.csv', index=False)
print('Inference demo complete → outputs/ranking_inference_top10.csv')



In [ ]:
# 8) Summary & Links
from pathlib import Path

def exists(p):
    return Path(p).exists()

summary = {
    'rf_metrics': 'outputs/rf_rank_metrics.json',
    'rf_top10': 'outputs/rf_rank_predictions_top10.csv',
    'clustering_optimal': 'outputs/clustering_optimal_model.json',
    'clustering_validation': 'outputs/clustering_validation.json',
    'clustering_stability': 'outputs/clustering_stability.json',
    'cluster_profiles': 'outputs/cluster_profiles.json',
    'xgb_tuned_metrics': 'outputs/ranking_metrics_tuned.json',
    'xgb_tuned_top10': 'outputs/ranking_predictions_top10_tuned.csv',
    'optuna_results': 'outputs/ranking_optimization_results.json',
    'production_model': 'models/xgb_ranker_production.pkl',
    'inference_demo': 'outputs/ranking_inference_top10.csv',
}

for k, p in summary.items():
    print(f"{k:24} -> {p} {'[OK]' if exists(p) else '[MISSING]'}")

